### Create Gaussian Process around dataset

In [1]:

from agent.components.GaussianProcess import get_empirical_variable_bounds
from agent.components.RASK import get_dependent_variable_mapping

%load_ext autoreload
%autoreload 2

import pandas as pd

from matplotlib import pyplot as plt
from agent.components.SLORegistry_v2 import calculate_weighted_SLO_F
from agent.components import RASK

from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType, theoretical_param_bounds, ServiceVar

from agent.components.commons import SLO_DEFAULT, SLO_HIGH_PERF, SLO_LOW_COST, SLO_HIGH_QUALITY

# s_type = ServiceType.QR
df = pd.read_csv("../statics/agent_experience/metrics_ICSOC_EXPLORE.csv")
df_preprocessed = RASK.preprocess_data(df)

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SLO_DEFAULT, SLO_HIGH_PERF, SLO_LOW_COST, SLO_HIGH_QUALITY]

INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str


### Gaussian Process gives you µ,s for an arbitrary point

In [2]:

# gp.predict(s_type, "max_tp", {'data_quality': 100, 'cores': 6.0})
# gp.predict(s_type, "max_tp", {'data_quality': 10000, 'cores': 600.0})
# gp.predict(s_type, "max_tp", {'data_quality': 100, 'cores': 6.0, 'model_size': 2.0})


### Test how much the epsilon should be moved each iteration

In [3]:
# from agent.components.commons import ServiceVar
# from agent.components.Optimizer import local_obj
# from agent.components.GaussianProcess import get_empirical_variable_bounds
# import numpy as np
#
# # Convert to a numpy array so we can do math on the whole vector
# x_norm = np.array([0.1] * (3 if s_type == ServiceType.CV else 2))
#
# simple_param_bounds = get_empirical_variable_bounds(gp_service.training_data)[s_type]
# del simple_param_bounds[ServiceVar.PERFORMANCE]
# simple_param_bounds = list(simple_param_bounds.values())
#
# for e in [1e-5, 1e-3, 1e-2, 5e-2]:
#     # val_start uses the original center
#     val_start = local_obj(x_norm, s_type, slos_default, gp_service, simple_param_bounds)
#
#     # x_norm + e now adds 'e' to every element (e.g., [0.11, 0.11])
#     val_nudge = local_obj(x_norm + e, s_type, slos_default, gp_service, simple_param_bounds)
#
#     diff = abs(val_start - val_nudge)
#     print(f"Eps {e}: Change in SLO-F is {diff:.6f}")

### Create versatile map of different solutions

In [4]:
# extract_pfo_for_SLOs(gp_service, slos_high_perf, "high_perf")

In [5]:
# extract_pfo_for_SLOs(gp_service, slos_low_cost, "low_cost")

In [6]:
# extract_pfo_for_SLOs(gp_service, slos_high_quality, "high_quality")

In [2]:

gp_list = []
lml_history = []

data_splits = 100
tested_range = 5

for i in range(tested_range):
    gp_all_services = {}

    for s in services:
        data_ratio = (i + 1) / data_splits

        # Initialize and train GP model
        _gp = GASK(s, show_figures=False)
        _gp.init_model(df, data_density=data_ratio)

        lml = _gp.get_model_lml(s, "max_tp")
        lml_history.append(lml)
        print(f"Ratio {data_ratio}: LML = {lml:.2f}")
        gp_all_services[s] = _gp

    gp_list.append(gp_all_services)


INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-qr-detector - Target: max_tp
/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__sigma_0 is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
INFO:multiscale:train_gp_models took 22 ms to execute
INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-cv-analyzer - Target: max_tp
/home/boris/development/composable-autonomous-offeri

Ratio 0.01: LML = -15.25
Ratio 0.01: LML = -19.94
Ratio 0.01: LML = -17.88
Ratio 0.02: LML = -21.44


/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__sigma_0 is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
INFO:multiscale:train_gp_models took 67 ms to execute
INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-pc-visualizer - Target: max_tp
INFO:multiscale:train_gp_models took 60 ms to execute
INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-qr-detector - Target: max

Ratio 0.02: LML = -37.98
Ratio 0.02: LML = -23.60


/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__sigma_0 is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
INFO:multiscale:train_gp_models took 172 ms to execute
INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-cv-analyzer - Target: max_tp
/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__sigma_0 is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a bett

...

Ratio 0.04: LML = -24.52
Ratio 0.04: LML = -74.47


INFO:multiscale:train_gp_models took 125 ms to execute
INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-qr-detector - Target: max_tp
/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__sigma_0 is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
INFO:multiscale:train_gp_models took 189 ms to execute


Ratio 0.04: LML = -31.30
Ratio 0.05: LML = -27.31


INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-cv-analyzer - Target: max_tp
/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__sigma_0 is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
INFO:multiscale:train_gp_models took 81 ms to execute
INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-pc-visualizer - Target: max_tp
INFO:multiscale:train_gp_models took 44 ms to exec

Ratio 0.05: LML = -97.02
Ratio 0.05: LML = -34.40


In [6]:
from agent.components.Optimizer import run_optimizer_multi
import itertools

solution_history = []

for i in range(tested_range):
    for s, q in itertools.product(services, slos):

        data_ratio = (i + 1) / tested_range
        gp = gp_list[i][s]

        # Run optimizer to find the best configuration
        solutions = run_optimizer_multi(s, q.slos, gp, theoretical_param_bounds[s], runs=10)
        fitness, config = max(solutions, key=lambda x: x[0])

        # Predict performance (mu, sigma) for the chosen configuration
        x_state = {ServiceVar.COST: config[0], ServiceVar.QUALITY: config[1]}
        x_state = x_state | ({ServiceVar.MODEL: config[2]} if s == ServiceType.CV else {})
        mu, sigma = gp.predict(s, "max_tp", x_state)

        # Store everything needed for the next block
        # We include empirical_var_bounds here as it changes per iteration
        solution_history.append({
            'data_rate': data_ratio,
            'rep': None,
            'service_type': s.name,
            'slo_set': q.name,
            'p_fitness': fitness,
            # 'config': config,
            # 'dist': (mu, sigma),
            'cores': x_state[ServiceVar.COST],
            'data_quality': x_state[ServiceVar.QUALITY],
            'model_size': x_state[ServiceVar.MODEL] if s == ServiceType.CV else -1,
            # 'x_state': x_state,
        })

        print(f"Optimal fitness for {s.value} and {q.name} with {(i + 1) * 20}% data: {fitness}")

Optimal fitness for elastic-workbench-qr-detector and default with 20% data: 0.39714488419877414
Optimal fitness for elastic-workbench-qr-detector and high_perf with 20% data: 0.7820760263595496
Optimal fitness for elastic-workbench-qr-detector and low_cost with 20% data: 0.8703179258137714
Optimal fitness for elastic-workbench-qr-detector and high_quality with 20% data: 0.6355404840542838
Optimal fitness for elastic-workbench-cv-analyzer and default with 20% data: 0.4436724753491919
Optimal fitness for elastic-workbench-cv-analyzer and high_perf with 20% data: 0.37217006448484224
Optimal fitness for elastic-workbench-cv-analyzer and low_cost with 20% data: 0.8855342773094317
Optimal fitness for elastic-workbench-cv-analyzer and high_quality with 20% data: 0.49668644185120725
Optimal fitness for elastic-workbench-pc-visualizer and default with 20% data: 0.738749447147898
Optimal fitness for elastic-workbench-pc-visualizer and high_perf with 20% data: 0.9312430134519571
Optimal fitness 

In [4]:

repeatable_data = []
runs_per_config = 5  # Cam be 5 as well

for row in solution_history:
    for i in range(runs_per_config):
        row['rep'] = i + 1
        repeatable_data.append(row.copy())

df_run = pd.DataFrame(repeatable_data)
df_run.to_csv('../statics/candidates/candidate_solutions.csv', index=False)

In [ ]:
# from agent.components.SLORegistry_v2 import calculate_weighted_SLO_F
# import numpy as np
#
# slo_means = []
# slo_stds = []
# num_samples = 500
#
# for entry in solution_history:
#     mu, sigma = entry['dist']
#     x_state = entry['x_state']
#     bounds = entry['bounds']
#
#     # TODO: This is no fair comparison, because it does not penalize the std
#     # 1. Sample performance points from the normal distribution
#     samples = np.random.normal(mu, sigma, num_samples)
#
#     # 2. Evaluate SLO fulfillment for every sample
#     slo_results = []
#     for s in samples:
#         sample_tp = {ServiceVar.PERFORMANCE: s}
#
#         val = calculate_weighted_SLO_F(
#             x_state | sample_tp,
#             slo_selected,
#             bounds
#         )
#         slo_results.append(val)
#
#     # 3. Aggregate statistics
#     slo_means.append(np.mean(slo_results))
#     slo_stds.append(np.std(slo_results))
#
# # Prepare data for plotting
# slo_means = np.array(slo_means)
# slo_stds = np.array(slo_stds)
# lml_history = np.array(lml_history)

In [ ]:

# import matplotlib.pyplot as plt

# fitness_list = np.array([f for f, c, g in solution_history])
#
# process_list = [g for f, c, g in solution_history]
# uncertainty = np.array([(sd / 300) for m, sd in process_list])

# under_line = slo_means - slo_stds
# over_line = slo_means + slo_stds
#
# hacked_certainty = (3.0 + (lml_history / 50))
# data_normalizer = [i + 1 / tested_range for i in range(tested_range)]

# plt.plot(hacked_certainty / data_normalizer, label='Model Likelihood', color='red', linewidth=2)

# plt.plot(slo_means, label='Mean SLO Fulfillment', color='#1A5276', linewidth=2)
#
# plt.fill_between(
#     range(len(slo_means)),
#     slo_means - slo_stds,
#     slo_means + slo_stds,
#     color='#5DADE2',
#     alpha=0.3,
#     label=r'$\pm 1$ Std Dev'
# )
#
# plt.xlabel("Generation/Index")
# plt.ylabel("Fitness Value")
# plt.title(f"Fitness Evolution for Service {s_type.value}")
# plt.legend()
# # plt.ylim([-0.5, 1.0])
# plt.savefig(f"../figures/evolution_{s_type.value}.jpg")
# plt.show()

In [ ]:
# from utils import visualize_ndarray
# from agent.components.commons import ServiceVar
# from typing import Dict
# from agent.components.Optimizer import VersatileMapElites
#
#
# def extract_pfo_for_SLOs(gp_service, slos: Dict[ServiceVar, float], slo_type: str, simple_bounds):
#     v_me = VersatileMapElites(s_type, bins=10)
#
#     #  I'm getting the black cells because they are not explored.
#     #  What I can do is force all cells to be explored at least once,
#     #  or just run gradient descent for each cell multiple (like 5) times.
#     v_me.run_search(slos, gp_service, simple_bounds, iterations=1000)
#     visualize_ndarray(v_me.fitness_table, gp_service.s_type.value + "_" + slo_type)
#
#     # (2) Here I get n best solutions
#     # Get n solutions that are high-performing but far apart
#     diverse_set = v_me.get_diverse_set(n_solutions=10, versatility=0.1)
#     return diverse_set
#     # print("\n".join(f"Versatile Candidate: {x}" for x in diverse_set))
#
# final_empirical_bounds = get_empirical_variable_bounds(df_preprocessed)[s_type]
# simple_param_bounds = final_empirical_bounds.copy()
# del simple_param_bounds[ServiceVar.PERFORMANCE]
# simple_param_bounds = list(simple_param_bounds.values())
#
# candidate_solutions = []
# # (1) Here I give it increasingly more training data
# for i in range(tested_range):
#     candidates = extract_pfo_for_SLOs(gp_list[i], slo_selected, "default", simple_param_bounds)
#     candidate_solutions.append(candidates)

In [ ]:

# Flatten the nested list and expand the coordinates
# flattened_data = []
# config_runs = 5  # Cam be 5 as well
#
#
# for i in range(tested_range):
#     for s, q in itertools.product(services, slos):
#
#         b_cores = theoretical_param_bounds[s][ServiceVar.COST]
#         b_quality = theoretical_param_bounds[s][ServiceVar.QUALITY]
#         if s == ServiceType.CV:
#             b_model = theoretical_param_bounds[s][ServiceVar.MODEL]
#
# for gen_idx, generation in enumerate(candidate_solutions):
#     for solution in generation:
#         # Extract scaled coordinates
#         cores = solution['coord'][0]
#         quality = solution['coord'][1]
#
#         # Unscale using: scaled * (max - min) + min
#         unscaled_cores = cores * (b_cores[1] - b_cores[0]) + b_cores[0]
#         unscaled_quality = quality * (b_quality[1] - b_quality[0]) + b_quality[0]
#
#         row = {
#             'data_rate': gen_idx + 1,
#             'rep': None,
#             'p_fitness': solution['fitness'],
#             'cores': unscaled_cores,
#             'data_quality': unscaled_quality,
#             'model_size': -1
#         }
#
#         if s_type == ServiceType.CV:
#             model_size = solution['coord'][2]
#             row['model_size'] = model_size * (b_model[1] - b_model[0]) + b_model[0]
#
#         for i in range(0, config_runs):
#             row['rep'] = i + 1
#             flattened_data.append(row.copy())
#
# # Export to CSV
# df_solutions = pd.DataFrame(flattened_data)
# df_solutions.to_csv('../statics/candidates/candidate_solutions.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np

# Load your recorded metrics
df_metrics = pd.read_csv('../statics/agent_experience/metrics_ICSOC_evaluation.csv')
df_metrics = RASK.preprocess_data(df_metrics)

# Split into 5 chronological parts
splits = np.array_split(df_metrics, len(gp_list))

from sklearn.metrics import mean_squared_error

slo_rmse_history = []
# Use the full recorded dataset as the benchmark for "Ground Truth"
# or a separate test holdout if preferred
test_df = df_metrics.copy()

dep_vars = get_dependent_variable_mapping(s_type)["max_tp"]

# Ground Truth: Based on ACTUAL throughput recorded in CSV
actual_slo_f = []
for _, row in test_df.iterrows():

    x_state = {
        ServiceVar.COST: row['cores'],
        ServiceVar.QUALITY: row['data_quality'],
        ServiceVar.PERFORMANCE: row['throughput']
    }
    if s_type == ServiceType.CV:
        x_state[ServiceVar.MODEL] = row['model_size']

    # TODO: I must get the theoretical bounds here, not the final ones
    val = calculate_weighted_SLO_F(
        x_state,
        slo_selected,
        theoretical_param_bounds[s_type]
    )
    actual_slo_f.append(val)

Y_empirical_splits = np.array_split(actual_slo_f, len(gp_list))
X_test_splits = np.array_split(test_df[dep_vars], len(gp_list))

# Compute Prediction Error for each GP stage
for idx, val_gp in enumerate(gp_list):
    X_test = X_test_splits[idx]
    pred_slo_f = []

    for i, row in enumerate(X_test):
        x_state = {
            ServiceVar.COST: row[0],
            ServiceVar.QUALITY: row[1],
        }

        # TODO: Using the mean here is likely wrong....
        mu, sigma = val_gp.predict(s_type, "max_tp", x_state)
        x_state[ServiceVar.PERFORMANCE] = mu  # (mu - 1.645 * sigma)
        if s_type == ServiceType.CV:
            x_state[ServiceVar.MODEL] = row[2]

        # Calculate fulfillment based on GP's PREDICTED throughput
        val = calculate_weighted_SLO_F(
            x_state,
            slo_selected,
            final_empirical_bounds
        )
        pred_slo_f.append(val)

    # RMSE between Ground Truth (Real CSV performance) and Predicted (GP performance)
    rmse = np.sqrt(mean_squared_error(Y_empirical_splits[idx], pred_slo_f))
    slo_rmse_history.append(rmse)
    print(f"GP Part {idx + 1}/5 | Recorded-Data RMSE: {rmse:.4f}")

In [ ]:

# 1. Prepare static validation set (unseen data)
# We sample from the full df to have a consistent benchmark
test_df = df_preprocessed.sample(frac=0.2, random_state=35)
test_df = test_df[test_df['service_type'] == s_type.value]
dep_vars = get_dependent_variable_mapping(s_type)["max_tp"]
X_test = test_df[dep_vars].values
y_actual_perf = test_df["max_tp"].values

# 2. Calculate "Ground Truth" SLO Fulfillment once
# We use the empirical bounds from the most complete model (the last one)
final_bounds = get_empirical_variable_bounds(gp_list[-1].training_data)[s_type]
actual_slo_f = []

for i in range(len(test_df)):
    row = test_df.iloc[i]
    x_state = {ServiceVar.COST: row.get('cores'), ServiceVar.QUALITY: row.get('data_quality')}
    if s_type == ServiceType.CV: x_state[ServiceVar.MODEL] = row.get('model_size')

    val = calculate_weighted_SLO_F(x_state | {ServiceVar.PERFORMANCE: y_actual_perf[i]}, slo_selected, final_bounds)
    actual_slo_f.append(val)

# 3. Loop through gp_list to calculate Predicted SLO Fulfillment RMSE
slo_rmse_history = []

for idx, val_gp in enumerate(gp_list):
    pred_slo_f = []
    y_pred_perf = val_gp.models[s_type]["max_tp"].predict(X_test)

    for i in range(len(test_df)):
        row = test_df.iloc[i]
        x_state = {ServiceVar.COST: row.get('cores'), ServiceVar.QUALITY: row.get('data_quality')}
        if s_type == ServiceType.CV: x_state[ServiceVar.MODEL] = row.get('model_size')

        # Calculate fulfillment based on GP prediction
        val = calculate_weighted_SLO_F(x_state | {ServiceVar.PERFORMANCE: y_pred_perf[i]}, slo_selected, final_bounds)
        pred_slo_f.append(val)

    # Compute RMSE for this specific GP
    rmse = np.sqrt(mean_squared_error(actual_slo_f, pred_slo_f))
    slo_rmse_history.append(rmse)

    print(f"GP {idx + 1} (Density {(idx + 1) / len(gp_list):.1f}) | SLO RMSE: {rmse:.4f}")

# --- Plot the learning curve ---
plt.figure(figsize=(8, 4))
plt.plot(np.linspace(0.1, 1.0, len(slo_rmse_history)), slo_rmse_history, marker='o', color='purple')
plt.title("SLO Fulfillment Error with increasing ")
plt.xlabel("Data Ratio")
plt.ylabel("RMSE (SLO Fulfillment)")
plt.grid(True, alpha=0.3)
plt.show()